In [1]:
import sys
sys.path.append('../src')

In [3]:
from VCFParser import VCFParser

vcf_file = '../data/pgx.passed.vcf'
vcf_parser = VCFParser(vcf_file)

In [4]:
#Test ensembl annotations
from QC_gate import requires_pass_qc
from ensembl import get_ensembl_info, parse_ensembl_response

with vcf_parser as parsed:
    for variant in parsed:
        ensembl_info = get_ensembl_info(variant)
        parsed_info = parse_ensembl_response(ensembl_info)

KeyboardInterrupt: 

In [ ]:
#Test pharmgkb annotations
from QC_gate import requires_pass_qc
from ensembl import get_ensembl_info, parse_ensembl_response, parse_pharmgkb_response
from pharmgkb import get_pharmgkb_info

#I just want to test on the first passed variant, so I will break after the first iteration
#There is rate limiting on the APIs, so I don't want to make too many calls during testing but variants are few, so I will get the first 10
with vcf_parser as parsed:
    counter = 0
    for variant in parsed:
        if counter >= 10:
            break
        requires_pass_qc(variant)
        ensembl_info = get_ensembl_info(variant)
        parsed_ensembl_info = parse_ensembl_response(ensembl_info)
        pharmgkb_info = get_pharmgkb_info(parsed_ensembl_info)
        parsed_pharmgkb_info = parse_pharmgkb_response(pharmgkb_info)
        counter += 1
        break

#Check the PharmGKB information
print(parsed_pharmgkb_info)

[{'id': 1450945340, 'drug': ['glucocorticoids'], 'isAssociated': True, 'significance': {'id': 769229666, 'resource': 'Association Significance', 'synonyms': [], 'term': 'yes', 'termId': 'variantAnnotationSignificance:769229666', 'valid': True}, 'evidence': 15104022}, {'id': 1448423632, 'drug': ['hydrochlorothiazide'], 'isAssociated': True, 'significance': {'id': 769229666, 'resource': 'Association Significance', 'synonyms': [], 'term': 'yes', 'termId': 'variantAnnotationSignificance:769229666', 'valid': True}, 'evidence': 15098080}, {'id': 1452328887, 'drug': ['abiraterone'], 'isAssociated': False, 'significance': {'id': 769229667, 'resource': 'Association Significance', 'synonyms': [], 'term': 'no', 'termId': 'variantAnnotationSignificance:769229667', 'valid': True}, 'evidence': 15143622}, {'id': 1452328843, 'drug': ['abiraterone'], 'isAssociated': True, 'significance': {'id': 769229666, 'resource': 'Association Significance', 'synonyms': [], 'term': 'yes', 'termId': 'variantAnnotatio

In [4]:
from pharmgkb import get_pharmgkb_info

#Code above didn't do as I expected, so I will force test with mock data of a known variant that is in PharmGKB - ACE
variant = type('Variant', (object,), {'CHROM': 'chr17', 'POS': 61554422})()  # Mock variant with known chromosome and position
parsed_info = {'gene_symbol': 'ACE'}  # Mock parsed info with known gene
pharmgkb_info = get_pharmgkb_info(parsed_info)
print(pharmgkb_info)

[{'id': 1185012329, 'drug': ['tacrolimus'], 'isAssociated': False, 'significance': {'id': 769229667, 'resource': 'Association Significance', 'synonyms': [], 'term': 'no', 'termId': 'variantAnnotationSignificance:769229667', 'valid': True}, 'evidence': 15083695}, {'id': 1018613244, 'drug': ['quinapril'], 'isAssociated': True, 'significance': {'id': 769229666, 'resource': 'Association Significance', 'synonyms': [], 'term': 'yes', 'termId': 'variantAnnotationSignificance:769229666', 'valid': True}, 'evidence': 11394446}, {'id': 1183491887, 'drug': ['lisinopril'], 'isAssociated': True, 'significance': {'id': 769229667, 'resource': 'Association Significance', 'synonyms': [], 'term': 'no', 'termId': 'variantAnnotationSignificance:769229667', 'valid': True}, 'evidence': 878189}, {'id': 982044530, 'drug': ['captopril'], 'isAssociated': True, 'significance': {'id': 769229666, 'resource': 'Association Significance', 'synonyms': [], 'term': 'yes', 'termId': 'variantAnnotationSignificance:76922966

In [7]:
len(pharmgkb_info)

5

In [10]:
from interpreter import LLMInterpreter, build_content

content = build_content(variant, parsed_info, pharmgkb_info)
print(content)  # verify the prompt looks right first



Variant: chr1:119507310:C/A
Gene: HSD3B1
Consequence: intron_variant
Impact: MODIFIER
Amino Acid Change: None
Allele Frequency: 0.013
PharmGKB annotations: [{'id': 1450945340, 'drug': ['glucocorticoids'], 'isAssociated': True, 'significance': {'id': 769229666, 'resource': 'Association Significance', 'synonyms': [], 'term': 'yes', 'termId': 'variantAnnotationSignificance:769229666', 'valid': True}, 'evidence': 15104022}, {'id': 1448423632, 'drug': ['hydrochlorothiazide'], 'isAssociated': True, 'significance': {'id': 769229666, 'resource': 'Association Significance', 'synonyms': [], 'term': 'yes', 'termId': 'variantAnnotationSignificance:769229666', 'valid': True}, 'evidence': 15098080}, {'id': 1452328887, 'drug': ['abiraterone'], 'isAssociated': False, 'significance': {'id': 769229667, 'resource': 'Association Significance', 'synonyms': [], 'term': 'no', 'termId': 'variantAnnotationSignificance:769229667', 'valid': True}, 'evidence': 15143622}, {'id': 1452328843, 'drug': ['abiraterone'

In [11]:
LLMInterpreter(content)

Raw LLM response:
## Clinical Interpretation Report

### Variant Overview
| Field | Value |
|-------|-------|
| **Location** | chr1:119507310 C>A |
| **Gene** | HSD3B1 |
| **Consequence** | Intron variant |
| **Predicted Impact** | MODIFIER |
| **Allele Frequency** | 1.3% |

### Gene Context
**HSD3B1** (3-beta-Hydroxysteroid Dehydrogenase Type 1) encodes an enzyme involved in steroid hormone biosynthesis, playing a critical role in the conversion of steroid precursors. This gene is clinically relevant in hormone-dependent conditions, particularly prostate cancer treatment.

### Variant Consequence Analysis
This variant is classified as an **intron variant** with a **MODIFIER** impact prediction. Intronic variants typically do not directly alter protein coding sequences, though they may potentially affect splicing or regulatory elements. No amino acid change is associated with this variant.

### Pharmacogenomic Associations (PharmGKB)
The provided PharmGKB annotations indicate associati

("## Clinical Interpretation Report\n\n### Variant Overview\n| Field | Value |\n|-------|-------|\n| **Location** | chr1:119507310 C>A |\n| **Gene** | HSD3B1 |\n| **Consequence** | Intron variant |\n| **Predicted Impact** | MODIFIER |\n| **Allele Frequency** | 1.3% |\n\n### Gene Context\n**HSD3B1** (3-beta-Hydroxysteroid Dehydrogenase Type 1) encodes an enzyme involved in steroid hormone biosynthesis, playing a critical role in the conversion of steroid precursors. This gene is clinically relevant in hormone-dependent conditions, particularly prostate cancer treatment.\n\n### Variant Consequence Analysis\nThis variant is classified as an **intron variant** with a **MODIFIER** impact prediction. Intronic variants typically do not directly alter protein coding sequences, though they may potentially affect splicing or regulatory elements. No amino acid change is associated with this variant.\n\n### Pharmacogenomic Associations (PharmGKB)\nThe provided PharmGKB annotations indicate associa